[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_02_mlp_forward/task_1_perceptron_and_activations.ipynb)

# Week 2 · Perceptron, activations and the MLP forward pass

Working in Colab? Replace `fiit-ba` in the URL with your GitHub username to open the copy in your fork, and run the setup cell below.

Last week you installed PyTorch. This week you use it only as a **tensor library**: you build the forward pass of a multi-layer perceptron (MLP) from scratch, inside a tiny framework whose structure mirrors `torch.nn`. No `torch.nn` layers, no autograd and no training yet; those come in the following weeks, on top of what you write today.

## What you will do
1. **Our tiny framework**: meet `Module`, the base class that stores layers in order (given).
2. **Shape convention**: one column per sample, `Z = W @ X + b`.
3. **Linear layer**: implement the forward pass of a fully-connected layer.
4. **Activation functions**: implement Sigmoid, Tanh, ReLU and LeakyReLU and check them against `torch`.
5. **Plot the activations**: see what the four functions look like.
6. **Perceptron**: one neuron, one sample, then a mini-batch.
7. **Model**: stack layers into an MLP and run a forward pass; cross-check against `torch.nn.Sequential`.
8. **Check your understanding**: three short questions.

## How to work through this notebook
- Every place that needs your input is marked with a `# TODO` comment and/or `...`. **Replace every `...` with your own code.**
- Written questions show *Your answer here.* Double-click the markdown cell and type your answer directly into it.
- Every implementation is followed by a **verification cell** that compares your code with PyTorch's built-in functions and ends with an `assert`. You are done with a part when it prints `OK` on every line; a `MISMATCH` stops the cell with an `AssertionError`, so a notebook that runs top to bottom without errors is a notebook whose implementation agrees with PyTorch.
- Run the cells **in order**: later cells depend on classes and variables defined earlier. If something looks broken, `Kernel -> Restart & Run All` is your friend.

*Adapted from the FIIT NSIETE course materials (vgg-fiit/NSIETE_2026).*

In [ ]:
# Colab setup (does nothing when you run the notebook locally)
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Everything this notebook needs (torch, matplotlib) is preinstalled in Colab; nothing to install.
    print("Running in Google Colab")

In [ ]:
%matplotlib inline
import copy
from collections import OrderedDict

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

print("torch", torch.__version__)

In [ ]:
# --- Configuration ---
SEED = 42          # fixed seed: everybody sees the same random weights and inputs

torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)

This week everything runs on the CPU in `torch.float64` (double precision), so that your results agree with PyTorch's to a few units of $10^{-16}$; the verification cells require an absolute difference below $10^{-10}$ (`atol=1e-10`, no relative tolerance). The tensors are tiny, a GPU would not help, and Apple's MPS backend does not support float64 anyway. We also never call `.backward()`: all tensors have `requires_grad=False`, and next week you will write the gradients yourself.

The verification cells use the small helper below: it prints one `OK` / `MISMATCH` line and returns the verdict, and the verification cells `assert` on it. Read it once; there is nothing to fill in.

In [ ]:
def check(name: str, ours, ref: torch.Tensor, atol: float = 1e-10) -> bool:
    """Compare `ours` with the reference tensor `ref`, print OK / MISMATCH and return the verdict."""
    if not isinstance(ours, torch.Tensor):
        print(f"{name:<40s} MISMATCH  (got {type(ours).__name__}, expected a torch.Tensor)")
        return False
    if ours.shape != ref.shape:
        print(f"{name:<40s} MISMATCH  (shape {tuple(ours.shape)}, expected {tuple(ref.shape)})")
        return False
    ours = ours.to(ref.dtype)
    ok = torch.allclose(ours, ref, rtol=0, atol=atol)
    max_diff = (ours - ref).abs().max().item()
    print(f"{name:<40s} {'OK' if ok else 'MISMATCH'}  (max |difference| = {max_diff:.1e})")
    return ok

## 1. Our tiny framework: `Module`

Every deep-learning framework has one elementary building block. In PyTorch it is `torch.nn.Module`: a layer, an activation function and a whole network are all modules, and a module can contain other modules. Our framework copies this design in miniature.

`Module` below does three things:
- it keeps its sub-modules in an `OrderedDict` called `modules`, so that a network remembers **in which order** its layers were added;
- `add_module(module, name)` registers a sub-module under a name, with the same sanity checks as PyTorch (no empty name, no `.` in the name, no clash with an existing attribute);
- `__call__` forwards to `forward`, so you can write `layer(X)` instead of `layer.forward(X)`, exactly as in PyTorch.

`forward` and `backward` are placeholders here. Each subclass implements its own `forward`; `backward` will hold the chain-rule derivatives next week. One small difference to remember: `torch.nn.Module.add_module` takes its arguments in the opposite order, `(name, module)`.

There is nothing to fill in here: run the two cells and continue.

In [ ]:
class Module:
    """Base class for every layer, activation and model in our tiny framework."""

    def __init__(self) -> None:
        self.modules: OrderedDict[str, "Module"] = OrderedDict()

    def add_module(self, module: "Module", name: str) -> None:
        if hasattr(self, name) and name not in self.modules:
            raise KeyError(f"attribute '{name}' already exists")
        elif "." in name:
            raise KeyError('module name can\'t contain "."')
        elif name == "":
            raise KeyError('module name can\'t be empty string ""')
        self.modules[name] = module

    def forward(self, *args, **kwargs) -> torch.Tensor:
        pass

    def backward(self, *args, **kwargs):
        pass

    def __call__(self, *args, **kwargs) -> torch.Tensor:
        return self.forward(*args, **kwargs)

In [ ]:
container = Module()
container.add_module(Module(), "first")
container.add_module(Module(), "second")
print("registered modules, in order:", list(container.modules))

for bad_name in ["", "with.dot", "forward"]:
    try:
        container.add_module(Module(), bad_name)
    except KeyError as e:
        print(f"add_module(..., {bad_name!r}) refused:", e)

# The same idea in PyTorch (note the (name, module) argument order):
seq = torch.nn.Sequential()
seq.add_module("fc1", torch.nn.Linear(3, 2))
seq.add_module("act1", torch.nn.ReLU())
print(seq)

## 2. Shape convention: one column per sample

Get the shapes right and everything else in the notebook falls into place.

In the lecture (and in this framework) a **mini-batch of $m$ samples with $n$ features each is a matrix $X$ of shape $(n, m)$: one column per sample.** A fully-connected (linear) layer with `in_features` $= n$ inputs and `out_features` $= k$ neurons has

- a weight matrix $W$ of shape $(k, n)$: row $j$ holds the weights of neuron $j$,
- a bias vector $b$ of shape $(k, 1)$: one number per neuron,

and computes

$$Z = W X + b, \qquad Z \in \mathbb{R}^{k \times m}.$$

In code: `Z = W @ X + b`. The matrix product $W X$ has shape $(k, n)\cdot(n, m) = (k, m)$. Adding $b$ of shape $(k, 1)$ **broadcasts** it over the $m$ columns, so every sample receives the same bias. Column $i$ of $Z$ is the output for sample $i$ and depends only on column $i$ of $X$: the samples in a batch never interact.

**`torch.nn` uses the transposed convention.** There a batch is `(m, n_features)`, one **row** per sample, and `torch.nn.Linear` computes `X @ W.T + b` with `W` of shape `(out_features, in_features)` and `b` of shape `(out_features,)`. The weight matrix has the same shape in both conventions; only the data is transposed. The verification cells below therefore transpose our inputs and outputs (`X.T`, `.T`) before comparing with PyTorch.

| | our framework (lecture) | `torch.nn` |
|---|---|---|
| batch of $m$ samples | `(n_features, m)` | `(m, n_features)` |
| weights `W` | `(out, in)` | `(out, in)` |
| bias `b` | `(out, 1)` | `(out,)` |
| forward | `W @ X + b` | `X @ W.T + b` |

**2.1 A `Linear(10, 3)` layer is applied to a mini-batch of 4 samples. What are the shapes of `X`, `W`, `b` and of the output `Z`? Which of these numbers does not appear in the parameter shapes at all, and why is that useful?**

_Your answer here._

## 3. The linear layer

In the lecture a perceptron was one neuron with one weight per input. In a framework, a whole layer of neurons is implemented with matrix algebra: `Linear` (also called *dense* or *fully-connected*) computes the pre-activations of all `out_features` neurons for all samples at once, with a single matrix product.

The layer and the activation function are kept in **separate modules**. That keeps the backward pass (chain rule) and the optimisation simple, as you will see next week.

The constructor is given: `W` is initialised with standard normal random numbers, `b` with zeros (question 8.3 asks why).

**3.1 Implement `Linear.forward`.**

Hints:
- `X` has shape `(in_features, m)`; the result must have shape `(out_features, m)`.
- `@` is matrix multiplication in PyTorch (`torch.matmul`); `*` is element-wise and is wrong here.
- Let broadcasting add the bias for you; there is no loop over samples.
- Leave `backward` as `pass`; you fill it in next week.

In [ ]:
class Linear(Module):
    """Fully-connected layer: Z = W @ X + b for a batch X of shape (in_features, m)."""

    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = torch.randn(out_features, in_features, dtype=torch.float64)   # (out, in), random init
        self.b = torch.zeros(out_features, 1, dtype=torch.float64)            # (out, 1), zero init

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # TODO: compute and return Z = W @ X + b    (X: (in_features, m)  ->  Z: (out_features, m))
        return ...

    def backward(self, dZ: torch.Tensor):
        pass   # next week: gradients dW, db and dX via the chain rule

**3.2 Verify against `torch.nn.functional.linear`.** PyTorch's `F.linear(input, weight, bias)` computes `input @ weight.T + bias` for row-per-sample batches, so we feed it `X.T`, give it the bias as a flat `(out,)` vector, and transpose the result back into our convention. The bias is overwritten with random numbers first: with the zero initialisation, a forgotten `+ b` would go unnoticed.

In [ ]:
torch.manual_seed(SEED)
lin = Linear(10, 3)
lin.b = torch.randn(3, 1, dtype=torch.float64)       # a non-zero bias, so that a forgotten '+ b' is caught
X = torch.randn(10, 4, dtype=torch.float64)          # 4 samples with 10 features each

Z_ours = lin(X)
Z_torch = F.linear(X.T, lin.W, lin.b.squeeze(1)).T   # torch.nn convention, transposed back

print("W:", tuple(lin.W.shape), " b:", tuple(lin.b.shape), " X:", tuple(X.shape))
assert check("Linear(10, 3) on a (10, 4) batch", Z_ours, Z_torch), "Linear.forward does not match F.linear: fix the TODO in 3.1"

## 4. Activation functions

A linear layer on its own can only compute affine functions of its input. **Activation functions** are applied element-wise to the pre-activations $Z$ and give the network its non-linearity. Four classics:

- **Sigmoid** squashes any real number into $(0, 1)$:
  $$\sigma(z) = \frac{1}{1 + e^{-z}}$$
- **Tanh** (hyperbolic tangent) squashes into $(-1, 1)$ and is zero-centred:
  $$\tanh(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}$$
- **ReLU** (rectified linear unit) keeps positive inputs and zeroes the rest; range $[0, \infty)$:
  $$\mathrm{ReLU}(z) = \max(0, z)$$
- **LeakyReLU** is ReLU with a small slope $\alpha$ (typically $0.01$) for negative inputs, so that no input is completely switched off; range $(-\infty, \infty)$:
  $$\mathrm{LeakyReLU}_\alpha(z) = \begin{cases} z & \text{if } z > 0 \\ \alpha z & \text{otherwise} \end{cases}$$

Each activation is a `Module` **without parameters**: `forward` maps a tensor to a tensor of the same shape.

**4.1 Implement the `forward` method of `Sigmoid`, `Tanh`, `ReLU` and `LeakyReLU`.**

Hints:
- Write the formulas yourself with `torch.exp`. Do not call `torch.sigmoid`, `torch.tanh` or `torch.relu`: they are what we compare against.
- ReLU without an `if`: `torch.clamp(Z, min=0)`, `torch.maximum(Z, torch.zeros_like(Z))` or `Z * (Z > 0)` all work on whole tensors.
- LeakyReLU: `torch.where(condition, value_if_true, value_if_false)` chooses element-wise between two tensors. The slope is stored in `self.alpha`.
- Leave every `backward` as `pass`; the derivatives are next week's topic.
- Side note: in the textbook Tanh formula, `exp` overflows to `inf` for $|z| \gtrsim 710$ in float64 (about $89$ in float32), and `inf / inf` is `nan`. `torch.tanh` is careful about this, which is one reason to prefer library functions in real code. For today the naive formula is fine.

In [ ]:
class Sigmoid(Module):
    def forward(self, Z: torch.Tensor) -> torch.Tensor:
        # TODO: return 1 / (1 + exp(-Z)), element-wise
        return ...

    def backward(self, dA: torch.Tensor):
        pass   # next week


class Tanh(Module):
    def forward(self, Z: torch.Tensor) -> torch.Tensor:
        # TODO: return (exp(Z) - exp(-Z)) / (exp(Z) + exp(-Z)), element-wise
        return ...

    def backward(self, dA: torch.Tensor):
        pass   # next week


class ReLU(Module):
    def forward(self, Z: torch.Tensor) -> torch.Tensor:
        # TODO: return max(0, Z), element-wise
        return ...

    def backward(self, dA: torch.Tensor):
        pass   # next week


class LeakyReLU(Module):
    def __init__(self, alpha: float = 0.01) -> None:
        super().__init__()
        self.alpha = alpha

    def forward(self, Z: torch.Tensor) -> torch.Tensor:
        # TODO: return Z where Z > 0 and alpha * Z elsewhere
        return ...

    def backward(self, dA: torch.Tensor):
        pass   # next week

**4.2 Verify against PyTorch** on nine points between $-4$ and $4$ (including $0$), and check that a 2-D input keeps its shape.

In [ ]:
z = torch.linspace(-4, 4, 9, dtype=torch.float64)        # -4, -3, ..., 3, 4
Z2 = torch.randn(3, 5, dtype=torch.float64)              # a (3, 5) "batch" of pre-activations

results = [
    check("Sigmoid vs torch.sigmoid", Sigmoid()(z), torch.sigmoid(z)),
    check("Tanh vs torch.tanh", Tanh()(z), torch.tanh(z)),
    check("ReLU vs torch.relu", ReLU()(z), torch.relu(z)),
    check("LeakyReLU(0.01) vs F.leaky_relu", LeakyReLU(alpha=0.01)(z), F.leaky_relu(z, negative_slope=0.01)),
    check("LeakyReLU(0.2) vs F.leaky_relu", LeakyReLU(alpha=0.2)(z), F.leaky_relu(z, negative_slope=0.2)),
    check("Sigmoid keeps a (3, 5) shape", Sigmoid()(Z2), torch.sigmoid(Z2)),
    check("ReLU keeps a (3, 5) shape", ReLU()(Z2), torch.relu(Z2)),
]
assert all(results), "some activations still MISMATCH: fix the TODOs in 4.1"
print("\nall activations OK")

**4.3 Why does an MLP need non-linear activations at all? What function does a network `Linear -> Linear -> Linear` *without* activations compute? Write it out for two layers.**

_Your answer here._

## 5. Plot the activations

**5.1 Compute the four activations on `x_plot` with your classes; the plotting code is given.**

Check that the curves match the formulas: Sigmoid squashes into $(0, 1)$, Tanh into $(-1, 1)$ and is zero-centred, ReLU is flat at $0$ for negative inputs, LeakyReLU keeps a small slope there.

In [ ]:
x_plot = torch.linspace(-4, 4, 100, dtype=torch.float64)

# TODO: compute the four activation outputs on x_plot using YOUR classes
y_sigmoid = ...
y_tanh = ...
y_relu = ...
y_leaky = ...

curves = [("Sigmoid", y_sigmoid), ("Tanh", y_tanh), ("ReLU", y_relu), ("LeakyReLU (alpha = 0.01)", y_leaky)]

# Verification: the curves must be the ones PyTorch would draw
references = [torch.sigmoid(x_plot), torch.tanh(x_plot), torch.relu(x_plot), F.leaky_relu(x_plot, negative_slope=0.01)]
curve_checks = [check(f"{name} on x_plot", y, ref) for (name, y), ref in zip(curves, references)]
assert all(curve_checks), "the curves do not match torch: use YOUR activation classes on x_plot"

fig, axs = plt.subplots(2, 2, figsize=(10, 7))
for ax, (name, y) in zip(axs.flat, curves):
    ax.plot(x_plot, y, linewidth=2)
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.axvline(0, color="gray", linewidth=0.7)
    ax.set_title(name)
    ax.set_xlabel("z")
    ax.set_ylabel("activation(z)")
    ax.grid(True, alpha=0.3)
fig.suptitle("Activation functions")
plt.tight_layout()
plt.show()

**5.2 Look at the plots. For which inputs does each function become (almost) flat? Why could a flat region be a problem once we start training (next week)?**

_Your answer here._

## 6. Perceptron forward pass

A **perceptron** is a single neuron: a weighted sum of the inputs plus a bias, followed by an activation function $\phi$,

$$a = \phi\!\left(\sum_{i=1}^{n} w_i x_i + b\right) = \phi\left(w^\top x + b\right).$$

In our framework this is `Linear(n, 1)` followed by an activation module: `W` is a `(1, n)` row vector and `b` a single number stored in a `(1, 1)` tensor. Two inputs are prepared below, both with $n = 10$ features:
- `xInput1` is **one sample**: shape `(10, 1)`, one column, features $0, 1, \dots, 9$.
- `xInput2` is a **mini-batch of 4 samples**: shape `(10, 4)`, four columns of random features.

**6.1 Create a perceptron with 10 inputs and one neuron, feed it both inputs and print the outputs together with their shapes.**

Hints:
- Nothing new to implement: use your `Linear` class.
- Expected shapes: `(1, 1)` for `xInput1` and `(1, 4)` for `xInput2`, one number per sample.

In [ ]:
# xInput1 is a single sample: 10 features in one column
xInput1 = torch.arange(10, dtype=torch.float64).reshape(10, 1)   # shape (10, 1)

# xInput2 is a mini-batch: 10 features, 4 samples (one per column)
xInput2 = torch.randn(10, 4, dtype=torch.float64)                # shape (10, 4)

# TODO: create the perceptron (Linear with 10 inputs and 1 neuron) and feed both inputs
perceptron = ...
z1 = ...        # output for xInput1
z2 = ...        # output for xInput2

print("z1 =", z1, "   shape:", tuple(z1.shape))
print("z2 =", z2, "   shape:", tuple(z2.shape))

**6.2 Verification.** One output per sample, and the samples in the mini-batch do not influence each other: column 2 of `z2` must equal the output of sample 2 on its own. A copy of the perceptron with a non-zero bias is compared with `F.linear` (a copy, so that `perceptron` itself keeps the bias that produced `z1` and `z2`).

In [ ]:
assert isinstance(z1, torch.Tensor) and z1.shape == (1, 1), f"z1 should have shape (1, 1), got {getattr(z1, 'shape', type(z1))}"
assert isinstance(z2, torch.Tensor) and z2.shape == (1, 4), f"z2 should have shape (1, 4), got {getattr(z2, 'shape', type(z2))}"
assert perceptron.W.shape == (1, 10) and perceptron.b.shape == (1, 1), "the perceptron should be Linear(10, 1)"
assert torch.allclose(z1, F.linear(xInput1.T, perceptron.W, perceptron.b.squeeze(1)).T), "z1 does not match torch.nn.functional.linear on xInput1"
assert torch.allclose(z2, F.linear(xInput2.T, perceptron.W, perceptron.b.squeeze(1)).T), "z2 does not match torch.nn.functional.linear on xInput2"
assert torch.allclose(z2[:, 2:3], perceptron(xInput2[:, 2:3])), "column 2 of z2 should be the output of sample 2 alone"
biased = copy.copy(perceptron)                             # a copy with a non-zero bias, so that a forgotten '+ b' is caught
biased.b = torch.randn(1, 1, dtype=torch.float64)
assert torch.allclose(biased(xInput2), F.linear(xInput2.T, biased.W, biased.b.squeeze(1)).T), "with a non-zero bias the perceptron no longer matches F.linear: is '+ b' missing?"
print("OK: shapes (1, 1) and (1, 4), both match F.linear, and every column is one independent sample")

**6.3 Add the activation: feed `z1` and `z2` through `Sigmoid` and, separately, through `LeakyReLU`, and print the four results.**

In a framework the activation is a separate module applied to the output of the linear layer, `a = activation(linear(x))`, so a perceptron with activation is just two modules in a row.

In [ ]:
# TODO: create a Sigmoid and a LeakyReLU module and apply each of them to z1 and to z2
sigmoid = ...
leaky_relu = ...
a_sigmoid1 = ...     # Sigmoid applied to z1
a_sigmoid2 = ...     # Sigmoid applied to z2
a_leaky1 = ...       # LeakyReLU applied to z1
a_leaky2 = ...       # LeakyReLU applied to z2

print("Sigmoid(z1)   =", a_sigmoid1)
print("Sigmoid(z2)   =", a_sigmoid2)
print("LeakyReLU(z1) =", a_leaky1)
print("LeakyReLU(z2) =", a_leaky2)

In [ ]:
results = [
    check("Sigmoid(z1) vs torch.sigmoid", a_sigmoid1, torch.sigmoid(z1)),
    check("Sigmoid(z2) vs torch.sigmoid", a_sigmoid2, torch.sigmoid(z2)),
    check("LeakyReLU(z1) vs F.leaky_relu", a_leaky1, F.leaky_relu(z1, negative_slope=0.01)),
    check("LeakyReLU(z2) vs F.leaky_relu", a_leaky2, F.leaky_relu(z2, negative_slope=0.01)),
]
assert all(results), "the activations of the perceptron output do not match torch: fix the TODO in 6.3"
if all(results):
    print(f"\nSigmoid outputs lie in   [{a_sigmoid2.min().item():.4f}, {a_sigmoid2.max().item():.4f}]")
    print(f"LeakyReLU outputs lie in [{a_leaky2.min().item():.4f}, {a_leaky2.max().item():.4f}]")

**6.4 Which range of values can the sigmoid perceptron produce, and which range the LeakyReLU perceptron? In which situation would you put a sigmoid on the output neuron?**

_Your answer here._

## 7. Model: stacking layers into an MLP

A **multi-layer perceptron** is a chain of modules in which the output of one module is the input of the next:

$$A^{[0]} = X, \qquad Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}, \qquad A^{[l]} = \phi^{[l]}\left(Z^{[l]}\right), \qquad \hat{Y} = A^{[L]}.$$

`Model` is a `Module` whose sub-modules are the layers and activations, registered with `add_module` in the order in which they should be applied. Because `self.modules` is an `OrderedDict`, iterating over it visits the modules in exactly that order.

**7.1 Implement `Model.forward`: pass the input through every registered module, in order, and return the final output.**

Hints:
- `for name, module in self.modules.items():` visits the modules in insertion order.
- Every module is callable: `A = module(A)`.
- The order matters; next week's backward pass will walk the same list in reverse.

In [ ]:
class Model(Module):
    """A sequential model: applies its registered modules one after another."""

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # TODO: feed X through every module in self.modules (in order) and return the result
        A = X
        for name, module in self.modules.items():
            ...
        return ...

    def backward(self, dA: torch.Tensor):
        pass   # next week: the same loop, in reverse

**7.2 Build a model for inputs with 16 features:**

`Linear(16, H1) -> Tanh -> Linear(H1, H2) -> ReLU -> Linear(H2, 1) -> Sigmoid`

Register each module with `model.add_module(module, name)` and a descriptive name (`"fc1"`, `"act1"`, `"fc2"`, ...); the first call is done for you. Then run a forward pass on the mini-batch `X` of 5 random samples and print the output and its shape.

Hints:
- The `in_features` of each `Linear` must equal the `out_features` of the previous one; activations do not change the shape.
- The final `Sigmoid` squashes the single output neuron into $(0, 1)$, like a probability for binary classification.
- Expected output shape: `(1, 5)`, one number per sample.

In [ ]:
H1, H2 = 32, 8                                   # sizes of the two hidden layers
X = torch.randn(16, 5, dtype=torch.float64)      # mini-batch: 16 features, 5 samples

model = Model()
# TODO: register the six modules in order: Linear(16, H1), Tanh, Linear(H1, H2), ReLU, Linear(H2, 1), Sigmoid
model.add_module(Linear(16, H1), "fc1")
...

Y = model(X)
print("modules:", [f"{name}: {type(m).__name__}" for name, m in model.modules.items()])
print("output:", Y)
print("output shape:", tuple(Y.shape))

**7.3 Verification.** The architecture, the output shape `(1, 5)` and the output range $(0, 1)$ are checked first. Then a copy of your model (`model` itself stays untouched) gets a random bias in every `Linear` (with the zero initialisation a forgotten `+ b` would go unnoticed), and we cross-check its numbers against `torch.nn.Sequential`: we build the same architecture from `torch.nn` modules, copy the `W` and `b` of the copy into it, and compare. Because `torch.nn` uses the row-per-sample convention we feed it `X.T` and transpose the result back. The copy happens inside `torch.no_grad()` because `torch.nn` parameters track gradients by default (we do not use that yet).

If some of your outputs sit very close to $0$ or $1$, that is not a bug: with 16 inputs and $\mathcal{N}(0, 1)$ weights the pre-activations are large and the sigmoid saturates. Question 8.3 comes back to this.

In [ ]:
expected_types = [Linear, Tanh, Linear, ReLU, Linear, Sigmoid]
actual_types = [type(m) for m in model.modules.values()]
assert actual_types == expected_types, \
    f"architecture MISMATCH: got {[t.__name__ for t in actual_types]}, expected Linear, Tanh, Linear, ReLU, Linear, Sigmoid"
print("architecture:   OK")

assert isinstance(Y, torch.Tensor), f"model(X) returned {type(Y).__name__} instead of a tensor: fix Model.forward"
assert tuple(Y.shape) == (1, 5), f"output shape MISMATCH: got {tuple(Y.shape)}, expected (1, 5)"
print("output shape:   OK")
assert bool(((Y >= 0) & (Y <= 1)).all()), "a Sigmoid output must lie in [0, 1]"
print("output in [0,1]: OK")

In [ ]:
# A copy of the model with non-zero biases, so that a forgotten '+ b' is caught (`model` itself stays untouched).
model_biased = copy.deepcopy(model)
ours_linears = [m for m in model_biased.modules.values() if isinstance(m, Linear)]
for m in ours_linears:
    m.b = torch.randn(m.out_features, 1, dtype=torch.float64)

# Build the torch.nn twin of the model, module by module, so it works for any architecture you registered.
TORCH_EQUIVALENT = {Sigmoid: torch.nn.Sigmoid, Tanh: torch.nn.Tanh, ReLU: torch.nn.ReLU}
layers = []
for m in model_biased.modules.values():
    if isinstance(m, Linear):
        layers.append(torch.nn.Linear(m.in_features, m.out_features, dtype=torch.float64))
    elif isinstance(m, LeakyReLU):
        layers.append(torch.nn.LeakyReLU(negative_slope=m.alpha))
    else:
        layers.append(TORCH_EQUIVALENT[type(m)]())
seq = torch.nn.Sequential(*layers)

# Copy our parameters into the torch layers: same W, bias flattened to (out,).
torch_linears = [m for m in seq if isinstance(m, torch.nn.Linear)]
with torch.no_grad():
    for ours, theirs in zip(ours_linears, torch_linears):
        theirs.weight.copy_(ours.W)
        theirs.bias.copy_(ours.b.squeeze(1))
    Y_torch = seq(X.T).T          # (5, 16) in, (5, 1) out, transposed back to (1, 5)

print(seq)
assert check("our Model vs torch.nn.Sequential", model_biased(X), Y_torch), "Model.forward does not match torch.nn.Sequential: fix the TODO in 7.1"

## 8. Check your understanding

**8.1 How many trainable parameters (weights and biases) does the model from 7.2 have with `H1 = 32` and `H2 = 8`? Give the formula per layer and the total. Do the activation modules add any?**

_Your answer here._

Confirm your answer in code:

In [ ]:
# TODO: count the parameters of `model` (all entries of W and b of every Linear layer; activations have none)
n_params = ...

n_torch = sum(p.numel() for p in seq.parameters())
print("parameters in our model:          ", n_params)
print("parameters in torch.nn.Sequential:", n_torch)
assert n_params == n_torch, f"parameter count MISMATCH: {n_params} != {n_torch} from torch.nn.Sequential"
print("OK")

**8.2 What happens if you forget the bias, i.e. the layer computes `Z = W @ X` only? Does the code crash? What can the model no longer do?**

_Your answer here._

**8.3 Why is `W` initialised with random numbers but `b` with zeros? What would go wrong if `W` were initialised with zeros (or any constant) as well?**

_Your answer here._

## Hand-in

Every hand-in in this course consists of the code (a commit in your fork) and, for anything that trains a model, a public Weights & Biases link. This week nothing is trained, so the checklist is short:

1. the notebook runs from top to bottom (`Kernel -> Restart & Run All`) without an `AssertionError` and every verification cell prints `OK`;
2. the written questions (2.1, 4.3, 5.2, 6.4, 8.1 to 8.3) are answered in the markdown cells;
3. the notebook is committed to your fork and you have sent the commit link, **by the end of the lab**.

Next week you add `backward` to every module you wrote today, implement loss functions and optimisers, and train an MLP with your own framework.